# Kaggle Playground Series 2026 — Customer Churn Prediction

**Goal:** Predict the probability of customer churn, evaluated on **ROC-AUC**.

### Pipeline Overview
1. Install & import libraries  
2. Load data  
3. Feature engineering  
4. Encoding  
5. Define models  
6. Cross-validated OOF predictions  
7. Ensemble & submission


## 1. Install & Import Libraries


In [4]:
# Uncomment to install if needed
# !pip install xgboost lightgbm catboost scikit-learn pandas numpy

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    HAS_XGB = True; print('XGBoost available')
except ImportError:
    HAS_XGB = False; print('XGBoost not found')

try:
    import lightgbm as lgb
    HAS_LGB = True; print('LightGBM available')
except ImportError:
    HAS_LGB = False; print('LightGBM not found')

try:
    import catboost as cb
    HAS_CB = True; print('CatBoost available')
except ImportError:
    HAS_CB = False; print('CatBoost not found')


XGBoost available
LightGBM available
CatBoost available


## 2. Load Data


In [5]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')
print(f'\nTarget distribution:')
print(train['Churn'].value_counts(normalize=True))


Train shape: (594194, 21)
Test  shape: (254655, 20)

Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [6]:
train.head()


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [7]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [8]:
train.describe()


,id,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000,594194.000000
mean,297096.500000,0.114102,36.577258,65.866223,2494.377057
std,171529.177262,0.317936,25.061922,31.067444,2353.916710
min,0.000000,0.000000,1.000000,18.250000,18.800000
25%,148548.250000,0.000000,12.000000,29.900000,639.650000
50%,297096.500000,0.000000,35.000000,74.100000,1433.650000
75%,445644.750000,0.000000,62.000000,90.800000,4263.800000
max,594193.000000,1.000000,72.000000,118.750000,8684.800000


## 3. Feature Engineering


In [9]:
def feature_engineering(df):
    df = df.copy()

 
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

    # average charge per month
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

    # charge normalised by number of active services
    service_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                    'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    svc_sum = df[service_cols].apply(
        lambda col: col.map({'Yes': 1, 'No': 0, 'No internet service': 0,
                             'No phone service': 0}).fillna(0)
    ).sum(axis=1)
    df['ChargePerService'] = df['MonthlyCharges'] / (svc_sum + 1)

    # tenure grouping
    df['TenureGroup'] = pd.cut(
        df['tenure'], bins=[0,12,24,48,72,np.inf], labels=[0,1,2,3,4]
    ).astype(int)
    df['IsNewCustomer']      = (df['tenure'] <= 3).astype(int)
    df['IsLongTermCustomer'] = (df['tenure'] >= 48).astype(int)

    # service bundle flags
    df['HasStreaming'] = (
        (df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')
    ).astype(int)
    df['HasSecurityServices'] = (
        (df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes') |
        (df['DeviceProtection'] == 'Yes') | (df['TechSupport'] == 'Yes')
    ).astype(int)

    # total add-on count
    addon_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                  'TechSupport','StreamingTV','StreamingMovies']
    df['NumAddOns'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)

    # interaction features
    df['MonthlyCharges_x_tenure'] = df['MonthlyCharges'] * df['tenure']
    df['IsMonthToMonth']          = (df['Contract'] == 'Month-to-month').astype(int)
    df['IsPaperlessElectronic']   = (
        (df['PaperlessBilling'] == 'Yes') &
        (df['PaymentMethod'] == 'Electronic check')
    ).astype(int)

    return df


train = feature_engineering(train)
test  = feature_engineering(test)
print(f'Train shape after FE: {train.shape}')


Train shape after FE: (594194, 32)


## 4. Encoding


In [10]:
target  = 'Churn'
id_test = test['id'].copy()

# binary-encode target
if train[target].dtype == object:
    train[target] = (train[target] == 'Yes').astype(int)

cat_cols = [
    'gender','Partner','Dependents','PhoneService','MultipleLines',
    'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies','Contract',
    'PaperlessBilling','PaymentMethod'
]

# combine train+test so encoding is consistent
combined = pd.concat(
    [train.drop(columns=[target]), test], axis=0, ignore_index=True
)
le = LabelEncoder()
for col in cat_cols:
    combined[col] = le.fit_transform(combined[col].astype(str))

n_train = len(train)
X       = combined.iloc[:n_train].drop(columns=['id'], errors='ignore').copy()
X_test  = combined.iloc[n_train:].drop(columns=['id'], errors='ignore').copy()
y       = train[target]

print(f'Feature count : {X.shape[1]}')
print(f'Features      : {list(X.columns)}')


Feature count : 30
Features      : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge', 'ChargePerService', 'TenureGroup', 'IsNewCustomer', 'IsLongTermCustomer', 'HasStreaming', 'HasSecurityServices', 'NumAddOns', 'MonthlyCharges_x_tenure', 'IsMonthToMonth', 'IsPaperlessElectronic']


## 5. Define Models


In [11]:
SEED    = 42
N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

models = {}

if HAS_XGB:
    models['xgb'] = xgb.XGBClassifier(
        n_estimators=800, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, gamma=0.1,
        reg_alpha=0.1, reg_lambda=1.0,
        eval_metric='auc', random_state=SEED, n_jobs=-1, verbosity=0
    )

if HAS_LGB:
    models['lgb'] = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.03, max_depth=6,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbose=-1
    )

if HAS_CB:
    models['cat'] = cb.CatBoostClassifier(
        iterations=600, learning_rate=0.05, depth=6,
        random_seed=SEED, verbose=0, eval_metric='AUC'
    )


models['gbm'] = GradientBoostingClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, random_state=SEED
)
models['rf'] = RandomForestClassifier(
    n_estimators=300, max_depth=10, random_state=SEED, n_jobs=-1
)

print(f'Models registered: {list(models.keys())}')


Models registered: ['xgb', 'lgb', 'cat', 'gbm', 'rf']


## 6. Cross-Validated OOF Predictions


In [12]:
oof_preds  = {name: np.zeros(len(X))      for name in models}
test_preds = {name: np.zeros(len(X_test)) for name in models}

for name, model in models.items():
    fold_auc = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model.fit(X_tr, y_tr)
        oof_preds[name][val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds[name] += model.predict_proba(X_test)[:, 1] / N_FOLDS

        fold_auc.append(roc_auc_score(y_val, oof_preds[name][val_idx]))

    cv_auc = roc_auc_score(y, oof_preds[name])
    print(f'  {name:6s}  folds: {[round(a,4) for a in fold_auc]}  -> OOF AUC: {cv_auc:.5f}')


  xgb     folds: [np.float64(0.916), np.float64(0.9171), np.float64(0.9166), np.float64(0.9176), np.float64(0.9149)]  -> OOF AUC: 0.91644
  lgb     folds: [np.float64(0.9159), np.float64(0.9169), np.float64(0.9163), np.float64(0.9175), np.float64(0.9147)]  -> OOF AUC: 0.91626
  cat     folds: [np.float64(0.9155), np.float64(0.9166), np.float64(0.916), np.float64(0.9171), np.float64(0.9144)]  -> OOF AUC: 0.91590
  gbm     folds: [np.float64(0.9157), np.float64(0.9167), np.float64(0.9162), np.float64(0.917), np.float64(0.9144)]  -> OOF AUC: 0.91596
  rf      folds: [np.float64(0.912), np.float64(0.9132), np.float64(0.9127), np.float64(0.9139), np.float64(0.9111)]  -> OOF AUC: 0.91258


## 7. Ensemble & Submission


In [13]:
# simple average ensemble
oof_blend  = np.mean(list(oof_preds.values()),  axis=0)
test_blend = np.mean(list(test_preds.values()), axis=0)

ensemble_auc = roc_auc_score(y, oof_blend)
print(f'Ensemble OOF AUC: {ensemble_auc:.5f}')


Ensemble OOF AUC: 0.91617


In [14]:
submission = pd.DataFrame({'id': id_test, 'Churn': test_blend})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved')
submission.head(10)


submission.csv saved!


,id,Churn
0,594194,0.063291
1,594195,0.001324
2,594196,0.093923
3,594197,0.003462
4,594198,0.462771
5,594199,0.168490
6,594200,0.900324
7,594201,0.002820
8,594202,0.032329
9,594203,0.314918


In [15]:
print('Churn probability stats:')
submission['Churn'].describe()


Churn probability stats:


count    254655.000000
mean          0.218274
std           0.272104
min           0.000954
25%           0.006865
50%           0.067475
75%           0.394078
max           0.976582
Name: Churn, dtype: float64